# road-damage-shift — S4 : référence zero-shot (Grounding DINO, Florence-2)

Deux détecteurs **sans aucun entraînement** sur RDD2022, guidés par une phrase par classe
(`configs/default.yaml`, section `zeroshot`, fixée avant tout résultat) :
`longitudinal crack` (D00), `transverse crack` (D10), `alligator crack` (D20), `pothole` (D40).

Comparés à YOLO11s (S1, seed 42) **sur les mêmes images** : test source et test États-Unis, mêmes métriques,
seuil du F1 choisi sur la validation source de chaque modèle. Deux lectures :
- **par classe** (protocole habituel) ;
- **« dégradation », toutes classes confondues** (NMS entre classes, IoU 0,7) : le modèle trouve-t-il les dégâts,
  même s'il se trompe de type ?

**Réglages Kaggle** : *Accelerator* = **GPU T4 x1**, Internet On, *Add Input* = le dataset des poids S1.
`SMOKE = True` d'abord (≈ 15 min, 50 images par jeu), puis `SMOKE = False` via *Save Version* (≈ 1 h 30).

In [ ]:
# ---- Paramètres -------------------------------------------------------------
REPO_URL = "https://github.com/<TON_USER>/road-damage-shift.git"  # à remplacer
REPO_REF = "main"
REPO_DIR = "/tmp/road-damage-shift"
RAW = "/tmp/rdd_raw"
DATA_OUT = "/tmp/rdd_yolo"
SMOKE = True                 # True : 50 premières images de chaque jeu
S1_MANIFEST_SHA256 = "aff1da5af0e1c61885e4a59ddc058d8f659638117f99ab5d1f5df525cc57e2da"
WEIGHTS = "/kaggle/input/datasets/<TON_USER>/rds-s1-weights/yolo11s_best.pt"  # poids S1 (fichier ou dossier)

In [ ]:
!rm -rf {REPO_DIR} && git clone -q --depth 1 -b {REPO_REF} {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" "transformers==5.24.1" faster-coco-eval
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
sys.path.insert(0, f"{REPO_DIR}/src")
import mlflow, pandas as pd, torch, transformers, ultralytics, yaml
from ultralytics import YOLO, settings
from roadshift.eval.compare import compare
from roadshift.eval.report import report
from roadshift.zeroshot.run import load_model, run as run_zeroshot
CFG = yaml.safe_load(open(f"{REPO_DIR}/configs/default.yaml"))
ZS, TRAIN = CFG["zeroshot"], CFG["train"]
GIT_SHA = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
VERSIONS = {"torch": torch.__version__, "ultralytics": ultralytics.__version__, "transformers": transformers.__version__,
            "gpu": torch.cuda.get_device_name(0)}
print(GIT_SHA[:10], VERSIONS)
print("prompts :", ZS["prompts"])
W = Path(WEIGHTS)
if W.is_dir():
    pts = sorted(W.rglob("*.pt")); assert len(pts) == 1, f"{W} contient {len(pts)} .pt"; W = pts[0]
W_SHA = hashlib.sha256(W.read_bytes()).hexdigest()
print("poids S1 :", W, W_SHA[:12])

## 0–1. Données (mêmes découpages qu'en S1, vérifiés par empreinte)

In [ ]:
!python -m roadshift.data.download --out {RAW} --countries Japan India Czech United_States
!python -m roadshift.data.convert --config {REPO_DIR}/configs/default.yaml --raw-root {RAW} --out {DATA_OUT} --overwrite
DATA = Path(DATA_OUT)
DATASET_INFO = json.load(open(DATA / "dataset_info.json"))
assert DATASET_INFO["manifest_sha256"] == S1_MANIFEST_SHA256, "Découpages différents de S1 : comparaison invalide"
EVAL = DATA
if SMOKE:  # mini-copie : 50 premières images de chaque liste, mêmes noms de fichiers
    EVAL = Path("/tmp/rdd_yolo_smoke"); shutil.rmtree(EVAL, ignore_errors=True); EVAL.mkdir()
    for d in ("images", "labels"):
        (EVAL / d).symlink_to(DATA / d)
    shutil.copy(DATA / "manifest.csv", EVAL)
    for f in DATA.glob("list_*.txt"):
        (EVAL / f.name).write_text("\n".join(f.read_text().split()[:50]) + "\n")
    for f in DATA.glob("data_*.yaml"):
        shutil.copy(f, EVAL)
TAGS = ZS["eval_sets"]
print("évaluation sur :", EVAL, TAGS)

## 2. Référence YOLO11s (S1), réévaluée ici : contrôle (doit redonner 0,368 et 0,412 en mAP@50)

In [ ]:
RUNS = Path("/kaggle/working/runs_zeroshot"); TIMES = {}
EVALS = {"source_val": ("data_source.yaml", "val"), "source_test": ("data_source.yaml", "test"),
         "test_United_States": ("data_eval_United_States.yaml", "val")}
t0 = time.time()
for tag in TAGS:
    yml, split = EVALS[tag]
    YOLO(str(W)).val(data=str(EVAL / yml), split=split, imgsz=TRAIN["imgsz"], batch=TRAIN["batch"],
                     conf=CFG["eval"]["conf_map"], save_json=True, plots=False, device=0, verbose=False,
                     project=str(RUNS), name=f"yolo11s/eval_{tag}", exist_ok=True)
TIMES["yolo11s"] = round(time.time() - t0, 1)
print(f"YOLO11s : {TIMES['yolo11s'] / 60:.1f} min")

## 3. Grounding DINO puis Florence-2 (un modèle à la fois en mémoire GPU)

In [ ]:
ZINFO = {}
for name in ("grounding_dino", "florence2"):
    t0 = time.time()
    loaded = load_model(name, ZS["models"][name], "cuda")
    ZINFO[name] = run_zeroshot(name, CFG, EVAL, TAGS, RUNS / name, "cuda", loaded=loaded)
    TIMES[name] = round(time.time() - t0, 1)
    print(f"{name} : {TIMES[name] / 60:.1f} min, révision {ZINFO[name]['revision']}")
    del loaded; torch.cuda.empty_cache()

## 4. Scores (IC bootstrap) et comparaisons appariées, par classe et toutes classes confondues

In [ ]:
MODELS = {"yolo11s": RUNS / "yolo11s", "grounding_dino": RUNS / "grounding_dino", "florence2": RUNS / "florence2"}
PAIRS = [("yolo11s", "grounding_dino"), ("yolo11s", "florence2"), ("grounding_dino", "florence2")]
NB = CFG["eval"]["bootstrap"]["n_resamples"]
SCORES = pd.concat([pd.DataFrame(report(d, EVAL, m, n_boot=NB, agnostic=a)) for a in (False, True)
                    for m, d in MODELS.items()])
CMP = pd.concat([pd.DataFrame(compare(MODELS, EVAL, ["source_test", "test_United_States"], "test_United_States",
                                      n_boot=NB, agnostic=a, pairs=PAIRS)) for a in (False, True)])
cols = ["model", "agnostic", "eval", "n_images", "conf_F1", "mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi"]
display(SCORES[cols])
display(CMP)
if not SMOKE:
    y = SCORES[(SCORES.model == "yolo11s") & ~SCORES.agnostic].set_index("eval")
    print("contrôle YOLO11s : source", round(y.loc["source_test", "mAP50"], 4), "(S1 : 0,368) | États-Unis",
          round(y.loc["test_United_States", "mAP50"], 4), "(S1 : 0,412)")

## 5. MLflow et sorties

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for k in ("MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if (v_ := _secret(k)):
        os.environ[k] = v_
os.chdir("/kaggle/working")
mlflow.set_tracking_uri(CFG["mlflow"]["tracking_uri"] or _secret("MLFLOW_TRACKING_URI") or "sqlite:////kaggle/working/mlflow.db")
mlflow.set_experiment(CFG["mlflow"]["experiment"] + "-zeroshot" + ("-smoke" if SMOKE else ""))
settings.update({"mlflow": False})

OUT = Path("/kaggle/working/results"); OUT.mkdir(exist_ok=True)
suffix = "_smoke" if SMOKE else ""
SCORES.to_csv(OUT / f"zeroshot_scores{suffix}.csv", index=False)
CMP.to_csv(OUT / f"zeroshot_comparisons{suffix}.csv", index=False)
(OUT / f"zeroshot_info{suffix}.json").write_text(json.dumps({"models": ZINFO, "seconds": TIMES}, indent=2))
with mlflow.start_run(run_name="zeroshot" + suffix):
    mlflow.set_tags({"stage": "S4-zeroshot", "smoke": SMOKE, "git_sha": GIT_SHA, "s1_weights_sha256": W_SHA,
                     "dataset_sha256": DATASET_INFO["manifest_sha256"], **VERSIONS,
                     **{f"{m}_revision": str(i["revision"]) for m, i in ZINFO.items()}})
    mlflow.log_params({"prompts": json.dumps(ZS["prompts"]), **{f"{m}_{k}": str(v) for m, mc in ZS["models"].items()
                                                                for k, v in mc.items()}})
    for _, r in SCORES.iterrows():
        mode = "agnostic" if r.agnostic else "by_class"
        mlflow.log_metrics({f"{r.model}/{mode}/{r['eval']}/{k}": float(r[k]) for k in ("mAP50", "F1")})
    mlflow.log_metrics({f"{m}/seconds": s for m, s in TIMES.items()})
    for f in OUT.glob(f"zeroshot_*{suffix}.*"):
        mlflow.log_artifact(str(f), artifact_path="results")

In [ ]:
# prédictions incluses (comparaisons appariées possibles plus tard, hors Kaggle)
!cd /kaggle/working && rm -f s4_zeroshot_outputs.zip && zip -q -9 -r s4_zeroshot_outputs.zip results mlflow.db mlruns runs_zeroshot -i "*.csv" "*.json" "*.db" "mlruns/*" && ls -lh s4_zeroshot_outputs.zip

## À me renvoyer
`s4_zeroshot_outputs.zip` (scores, comparaisons, prédictions des trois modèles, MLflow).